<a href="https://colab.research.google.com/github/arpitiitb/docs-rag/blob/main/05_retrieval_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steps 5-7: Measure and improve retrieval (about 2-3 hours, free)

**Where you are:** you have a RAG system (Steps 1-3) and an exam for it (Step 4). Now you grade the **retrieval** part and improve it:

| Step | System | Idea |
|---|---|---|
| 5 | **Dense** (baseline) | embeddings only, what you built in Step 2 |
| 6 | **Hybrid** | embeddings + keyword search, merged |
| 7 | **Hybrid + reranker** | a stronger model re-orders the top candidates |

**Why measure retrieval before answers?** If the right page never reaches the prompt, the LLM cannot answer correctly, however good the prompt is. Retrieval sets the ceiling, so you fix it first. (Step 8 grades the answers.)

**No OpenAI calls in this notebook,** so it costs nothing.


## 5.1 Setup

In [1]:
%pip install -q sentence-transformers faiss-cpu rank-bm25 tabulate
import os, re, json, time
import numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/docs-rag"
else:
    BASE = "/content/docs-rag"
DATA, RESULTS = f"{BASE}/data", f"{BASE}/results"
os.makedirs(RESULTS, exist_ok=True)

chunks = json.load(open(f"{DATA}/chunks.json"))
index = faiss.read_index(f"{DATA}/index.faiss")
meta = json.load(open(f"{DATA}/index_meta.json"))
emb = SentenceTransformer(meta["embedding_model"])
ts = json.load(open(f"{DATA}/testset.json"))
test = [q for q in ts["items"] if q["gold_url"]]          # retrieval needs a gold page
print(f"{len(chunks)} chunks | test set v{ts['version']}: {ts['n']} items, {len(test)} in scope for retrieval")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

175 chunks | test set v1: 21 items, 19 in scope for retrieval


## 5.2 The metrics

**Concepts**
- A retriever returns a ranked list of chunks. A result is a **hit** if a chunk from the gold page appears in it. (We grade at page level, because your labels are pages.)
- **recall@k:** the share of questions where a gold-page chunk appears in the top k. recall@5 = 0.8 means the right page reached the top 5 for 80% of questions. This matters most, because the top k is what goes into the prompt.
- **MRR@k (mean reciprocal rank):** rewards finding it *early*. Rank 1 scores 1, rank 2 scores 0.5, rank 3 scores 0.33, not found scores 0. Then average.
- recall@1 is the strictest: was the very first chunk from the right page?

Check the functions on a tiny example first, where you can work out the answer by hand.

In [2]:
def first_hit_rank(ids, gold_url, k):
    for rank, i in enumerate(ids[:k], start=1):
        if chunks[i]["url"] == gold_url:
            return rank
    return None

def score(results, items, ks=(1, 3, 5, 10)):
    out = {}
    for k in ks:
        out[f"recall@{k}"] = np.mean([first_hit_rank(r, q["gold_url"], k) is not None for r, q in zip(results, items)])
    ranks = [first_hit_rank(r, q["gold_url"], 5) for r, q in zip(results, items)]
    out["MRR@5"] = np.mean([1 / x if x else 0 for x in ranks])
    return out

# hand-checkable example: gold page found at rank 1, at rank 3, and not at all
toy_gold = chunks[0]["url"]
others = [i for i, c in enumerate(chunks) if c["url"] != toy_gold][:6]
toy_results = [[0] + others[:4], others[:2] + [0] + others[2:4], others[:5]]
print(score(toy_results, [{"gold_url": toy_gold}] * 3, ks=(1, 3, 5)))
# expected: recall@1 = 1/3, recall@3 = 2/3, recall@5 = 2/3, MRR@5 = (1 + 1/3 + 0) / 3 = 0.444

{'recall@1': np.float64(0.3333333333333333), 'recall@3': np.float64(0.6666666666666666), 'recall@5': np.float64(0.6666666666666666), 'MRR@5': np.float64(0.4444444444444444)}


## 5.3 Baseline: dense retrieval

We retrieve 10 chunks per question, which lets us compute every k at once, and also time each search. Latency matters in production: users notice anything over about a second.

In [3]:
QPREFIX = "Represent this sentence for searching relevant passages: "

def dense(q, k=10):
    qv = emb.encode([QPREFIX + q], normalize_embeddings=True).astype("float32")
    _, ids = index.search(qv, k)
    return [int(i) for i in ids[0]]

def run(name, retriever):
    t0 = time.perf_counter()
    res = [retriever(q["question"]) for q in test]
    ms = (time.perf_counter() - t0) / len(test) * 1000
    row = {"system": name, **score(res, test), "ms/query": round(ms, 1)}
    return row, res

rows, all_results = [], {}
row, all_results["dense"] = run("dense", dense)
rows.append(row)
pd.DataFrame(rows).round(3)

,system,recall@1,recall@3,recall@5,recall@10,MRR@5,ms/query
0,dense,0.579,0.789,0.842,0.895,0.68,67.4


**What you should see:** a single row. Typical baselines on a small docs corpus land around recall@5 of 0.6-0.9. A low number is not a failure, it's your "before" picture.

### Look at the misses

Numbers tell you *how much*, misses tell you *why*. For each question where the gold page is not in the top 5, compare what came back.

In [4]:
def show_misses(res, k=5):
    n = 0
    for r, q in zip(res, test):
        if first_hit_rank(r, q["gold_url"], k) is None:
            n += 1
            print(f"[{q['type']}] {q['question']}\n   gold: {q['title']}")
            for i in r[:3]:
                print(f"   got:  {chunks[i]['title']}")
            print()
    print(n, "misses in top", k)

show_misses(all_results["dense"])

[generated] What is the top-level container in Unity Catalog that holds all securable objects?
   gold: Unity Catalog setup guide | Databricks on AWS
   got:  Unity Catalog securable objects reference | Databricks on AWS
   got:  Unity Catalog securable objects reference | Databricks on AWS
   got:  Unity Catalog securable objects reference | Databricks on AWS

[paraphrase] Is there a central place to see my workspace assets and structure in the interface?
   gold: What is Catalog Explorer? | Databricks on AWS
   got:  Unity Catalog setup guide | Databricks on AWS
   got:  Managed versus external assets in Unity Catalog | Databricks on AWS
   got:  Upgrade a Databricks workspace to Unity Catalog | Databricks on AWS

[paraphrase] How do I authorize a specific user group to run queries using AI gateway endpoints?
   gold: Govern models and MCPs with GRANT policies | Databricks on AWS
   got:  Tutorial: Control a coding agent's model access with Unity Gateway | Databricks on AWS
   got:  

**Ask yourself for each miss:** is the retrieved page actually a reasonable answer too (a labeling issue), is the question worded very differently from the page (a dense-search weakness), or does it use an exact term (keyword search should help)? Write down 2-3 patterns.

### Results by question type

In [5]:
def by_type(res, k=5):
    df = pd.DataFrame({"type": [q["type"] for q in test],
                       "hit": [first_hit_rank(r, q["gold_url"], k) is not None for r, q in zip(res, test)]})
    return df.groupby("type")["hit"].agg(["mean", "count"]).rename(columns={"mean": f"recall@{k}", "count": "n"})

by_type(all_results["dense"]).round(2)

,recall@5,n
type,,
exact_term,1.00,3
generated,0.88,8
paraphrase,0.75,8


With only 3 exact-term questions, one hit or miss swings that row by 33 points. Read it as a direction, not a measurement.

## 6.1 Keyword search with BM25

**Concepts**
- **BM25** is the classic search-engine ranking formula. It scores a chunk higher when it contains the query's words, especially **rare** words (a word in every chunk, like "the", counts for almost nothing), and it stops rewarding a word after a few repeats.
- It knows nothing about meaning ("rejected" does not match "denied"), but it is excellent at **exact terms**: privilege names like `USE CATALOG`, error codes and product names, which embeddings can blur.
- Embeddings and BM25 make **different mistakes**, which is why combining them works.

In [6]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tokenize(c["title"] + " " + c["text"]) for c in chunks])

def sparse(q, k=10):
    scores = bm25.get_scores(tokenize(q))
    return [int(i) for i in np.argsort(scores)[::-1][:k]]

row, all_results["bm25"] = run("bm25", sparse)
rows.append(row)
pd.DataFrame(rows).round(3)

,system,recall@1,recall@3,recall@5,recall@10,MRR@5,ms/query
0,dense,0.579,0.789,0.842,0.895,0.680,67.4
1,bm25,0.632,0.789,0.789,0.842,0.702,1.1


**What you should see:** BM25 alone may be better or worse than dense overall. Check `by_type(all_results["bm25"])`: it often does well on exact-term questions and worse on paraphrases.

## 6.2 Hybrid search with reciprocal rank fusion (RRF)

**Concepts**
- We have two ranked lists. Their **scores can't be compared** (a cosine of 0.6 and a BM25 of 12.4 are on different scales), so we merge by **rank** instead.
- **RRF:** each chunk gets `1 / (60 + rank)` from each list, summed. A chunk ranked high in either list rises; one ranked high in both rises most. The 60 dampens the advantage of the very top positions, and is the common default.
- We take the top 30 from each retriever, so a chunk that one method ranks 25th can still be rescued by the other.

In [7]:
def rrf(lists, k=10, c=60):
    scores = {}
    for lst in lists:
        for rank, i in enumerate(lst, start=1):
            scores[i] = scores.get(i, 0) + 1 / (c + rank)
    return [i for i, _ in sorted(scores.items(), key=lambda x: -x[1])][:k]

def hybrid(q, k=10):
    return rrf([dense(q, 30), sparse(q, 30)], k)

row, all_results["hybrid"] = run("hybrid (dense + bm25, RRF)", hybrid)
rows.append(row)
pd.DataFrame(rows).round(3)

,system,recall@1,recall@3,recall@5,recall@10,MRR@5,ms/query
0,dense,0.579,0.789,0.842,0.895,0.680,67.4
1,bm25,0.632,0.789,0.789,0.842,0.702,1.1
2,"hybrid (dense + bm25, RRF)",0.684,0.789,0.842,0.895,0.741,78.3


**What you should see:** hybrid usually matches or beats the better of the two on recall@5. It is not guaranteed: on a small set it can lose a question or two. If it does, look at which ones with `show_misses(all_results["hybrid"])`; that is a finding, not a mistake.

## 7.1 Add a reranker

**Concepts**
- Your embedding model is a **bi-encoder**: it encodes the question and each chunk **separately**, then compares vectors. That is fast (chunk vectors are precomputed) but coarse.
- A **cross-encoder** reads the question and a chunk **together** and outputs one relevance score. It sees how the words of both interact, so it is much more accurate, but far slower, since it must run once per (question, chunk) pair at query time.
- **The standard pattern:** a fast retriever finds about 20 candidates, the cross-encoder re-orders them, and you keep the top 5. You get most of the accuracy for a fraction of the cost.
- The model `cross-encoder/ms-marco-MiniLM-L-6-v2` is small, free and runs on CPU. The first run downloads about 90 MB.

In [8]:
from sentence_transformers import CrossEncoder
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank(q, k=10, n_candidates=20):
    cand = rrf([dense(q, 30), sparse(q, 30)], n_candidates)
    scores = ce.predict([(q, chunks[i]["title"] + "\n" + chunks[i]["text"]) for i in cand])
    return [cand[j] for j in np.argsort(scores)[::-1][:k]]

row, all_results["hybrid+rerank"] = run("hybrid + rerank", hybrid_rerank)
rows.append(row)
pd.DataFrame(rows).round(3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,system,recall@1,recall@3,recall@5,recall@10,MRR@5,ms/query
0,dense,0.579,0.789,0.842,0.895,0.680,67.4
1,bm25,0.632,0.789,0.789,0.842,0.702,1.1
2,"hybrid (dense + bm25, RRF)",0.684,0.789,0.842,0.895,0.741,78.3
3,hybrid + rerank,0.684,0.789,0.895,0.947,0.763,8055.8


**What you should see:** the reranker mostly improves **recall@5 and MRR**. It cannot improve recall@20, because it only reorders the 20 candidates it is given. Note the **ms/query** column: the reranker is much slower. That trade-off is the point to make when you present this.

## 7.2 The comparison table


In [9]:
results = pd.DataFrame(rows)
results.insert(1, "test_set", f"v{ts['version']} ({len(test)} in-scope q)")
results.to_csv(f"{RESULTS}/retrieval_results.csv", index=False)
print(results.drop(columns="test_set").round(3).to_markdown(index=False))

types = pd.concat({name: by_type(res)[f"recall@5"] for name, res in all_results.items()}, axis=1)
types["n"] = by_type(all_results["dense"])["n"]
print("\nrecall@5 by question type:\n")
print(types.round(2).to_markdown())

| system                     |   recall@1 |   recall@3 |   recall@5 |   recall@10 |   MRR@5 |   ms/query |
|:---------------------------|-----------:|-----------:|-----------:|------------:|--------:|-----------:|
| dense                      |      0.579 |      0.789 |      0.842 |       0.895 |   0.68  |       67.4 |
| bm25                       |      0.632 |      0.789 |      0.789 |       0.842 |   0.702 |        1.1 |
| hybrid (dense + bm25, RRF) |      0.684 |      0.789 |      0.842 |       0.895 |   0.741 |       78.3 |
| hybrid + rerank            |      0.684 |      0.789 |      0.895 |       0.947 |   0.763 |     8055.8 |

recall@5 by question type:

| type       |   dense |   bm25 |   hybrid |   hybrid+rerank |   n |
|:-----------|--------:|-------:|---------:|----------------:|----:|
| exact_term |    1    |   1    |     1    |            1    |   3 |
| generated  |    0.88 |   0.75 |     0.88 |            0.88 |   8 |
| paraphrase |    0.75 |   0.75 |     0.75 |         

## 7.3 What is still failing?

Look at the misses of your best system. These feed Step 9 (error analysis).

In [10]:
best = max(all_results, key=lambda n: score(all_results[n], test)["recall@5"])
print("Best system by recall@5:", best, "\n")
show_misses(all_results[best])

Best system by recall@5: hybrid+rerank 

[generated] d
   gold: Unity Catalog securable objects reference | Databricks on AWS
   got:  Track model usage | Databricks on AWS
   got:  Track model usage | Databricks on AWS
   got:  Track model usage | Databricks on AWS

[paraphrase] How do I authorize a specific user group to run queries using AI gateway endpoints?
   gold: Govern models and MCPs with GRANT policies | Databricks on AWS
   got:  AI governance with Unity Gateway | Databricks on AWS
   got:  Track model usage | Databricks on AWS
   got:  Unity Gateway observability | Databricks on AWS

2 misses in top 5


These are good results. Here’s what they mean in plain terms. You have 19 questions, so each 0.053 is one question.

The story, step by step

Dense alone is a decent baseline: the right page reaches the top 5 for 16 of 19 questions, but it’s first only 11 times.
BM25 alone puts the right page first more often (12 times) but misses more overall (15 in the top 5). Keywords are precise but miss reworded questions.
Hybrid gets the best of both: the right page is first 13 times (up from 11) without losing anything in the top 5. MRR rises from 0.68 to 0.74, so the ordering got better.
The reranker found one more question in the top 5 (17 of 19) and had the best MRR (0.76). The by-type table shows where: it rescued one paraphrase question (6 of 8 became 7 of 8), which matches what rerankers are good at.

By question type

Exact term (3 questions): every system got all 3. The expected BM25 advantage didn’t show here because there were too few such questions.
Generated: dense and hybrid got 7 of 8, BM25 6 of 8.
Paraphrase: everything got 6 of 8 except the reranker, with 7 of 8.

The important catch: speed. The reranker took 8 seconds per question, about 100 times slower than hybrid, because Colab’s free CPU is slow at running the cross-encoder. That’s too slow for a real product. Ways to fix it, if you want a better number:

Switch Colab to a GPU (Runtime > Change runtime type > T4 GPU) and rerun 7.1-7.2, which should be far faster.
Or rerank 10 candidates instead of 20.

How to describe the results: “Hybrid search improved first-rank accuracy from 11 to 13 of 19 questions (MRR 0.68 to 0.74). A cross-encoder reranker raised recall@5 from 16 to 17 of 19 (MRR 0.76), but cost about 8 seconds per query on CPU, so I’d use it only with a GPU or fewer candidates.” Note that every improvement is one or two questions, so it’s a consistent trend rather than proof. Saying so is a strength in an interview.

**Next: Step 8, grade the answers (faithfulness and correctness).**